# SCM Data Agent 設定段階別 横断評価

L0〜L5 の 6 エージェントを 30 問の評価データセットで一括評価し、accept / reject / unclear 件数と accept 率、1 問あたりの平均応答時間を比較する。

> Microsoft Fabric のノートブック上で、既定 Lakehouse に `Files/scm_eval_questions.csv` を配置した状態で実行する想定。

In [ ]:
%pip install -q fabric-data-agent-sdk
import time
import pandas as pd
from fabric.dataagent.evaluation import evaluate_data_agent

# 30問の評価データセットを読み込む
eval_df = pd.read_csv("/lakehouse/default/Files/scm_eval_questions.csv")

agents = [
    "agent_scm_01_bad_naming",   # L0
    "agent_scm_01_good_naming",  # L1
    "agent_scm_02_logistics",    # L2
    "agent_scm_07_focused",      # L3
    "agent_scm_09_dsdocs",       # L4
    "agent_scm_10_examples",     # L5
]

scores = []
for a in agents:
    t0 = time.time()
    evaluate_data_agent(
        df=eval_df,
        data_agent_name=a,
        workspace_name="Book",
        table_name=f"eval_{a}",
    )
    elapsed = time.time() - t0
    res = spark.read.table(f"eval_{a}").toPandas()
    scores.append({
        "agent":      a,
        "accept":     (res.status == "accept").sum(),
        "reject":     (res.status == "reject").sum(),
        "unclear":    (res.status == "unclear").sum(),
        "rate":       (res.status == "accept").mean(),
        # 1問あたりの平均応答時間（秒）。レスポンス速度の比較指標
        "avg_latency_sec": elapsed / len(eval_df),
    })

pd.DataFrame(scores)

## 設定段階別の精度を可視化

accept / unclear / reject を積み上げ棒グラフにして、設定段階ごとの精度の伸びを可視化する。

In [ ]:
import matplotlib.pyplot as plt

df = pd.DataFrame(scores).set_index("agent")
df[["accept", "unclear", "reject"]].plot(
    kind="bar", stacked=True, figsize=(10, 5),
    color=["#2E7D32", "#FFC107", "#C62828"],
)
plt.title("SCM Data Agent — 設定段階別 精度比較 (n=30)")
plt.ylabel("質問数")
plt.legend(title="判定")
plt.tight_layout()
plt.show()